In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/C0PEP0D/SoftMobility.git
except ImportError:
    pass

# Trajectories, optimization and performance evaluation of a soft surfer in Taylor–Green vortices


### Imports

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax

import softmobility as sm
from softmobility.tutorials import figstyle

figstyle.apply()
np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

In [ ]:
# Drop the local colour constants in favour of the project palette.
BLUEACCENT = figstyle.COLORS["blue"]
REDACCENT = figstyle.COLORS["red"]
GREY = figstyle.COLORS["grey"]

## Simulation of default parameters

### YAML file

In [3]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:
  - spring_k
  - radius
input_names:
  - gravity
  - active_force

# Default Values (Optional)
defaults:
  spring_k: .5
  radius: .33
  _spr_: 50
  _rad_: 1

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: 1
    position: [0, 0, 1]
    orientation: [phi, 0, 0]
    force: [-gravity0, -gravity1, -gravity2]
    torque: [-_spr_ * spring_k * phi, 0, 0]

  - # Sphere 1 #################
    radius: _rad_ * radius
    position: [0, 0, -radius]
    orientation: [-phi/_rad_/radius, 0, 0]
    force: [gravity0,
             gravity1 + active_force * sin(phi/_rad_/radius),
             gravity2 + active_force * cos(phi/_rad_/radius)]
    torque: [_spr_ * spring_k * phi, 0, 0]
"""

In [4]:
yaml_data_rigid = """
# Variable Prefixes (Optional)
design_names:
  - radius
input_names:
  - gravity
  - active_force

# Default Values (Optional)
defaults:
  radius: .5
  _rad_: 1

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: 1
    position: [0, 0, 1]
    orientation: [0, 0, 0]
    force: [-gravity0, -gravity1, -gravity2]
    torque: [0, 0, 0]

  - # Sphere 1 #################
    radius: _rad_ * radius
    position: [0, 0, -radius]
    orientation: [0, 0, 0]
    force: [gravity0, gravity1, gravity2 + active_force]
    torque: [0, 0, 0]
"""

### Soft Body

In [5]:
mybody= sm.SoftBody(yaml_data, verbose=True)
myrigidbody = sm.SoftBody(yaml_data_rigid, verbose=False)

  Found variables: phi, radius, spring_k, gravity0, gravity1, gravity2, active_force
  3D field inputs:  ['gravity']
  Scalar inputs:    ['active_force']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '1']
      Orientation: ['phi', '0', '0']
      Coupling matrix C_H:
[['-1', '0', '0', '0'], ['0', '-1', '0', '0'], ['0', '0', '-1', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['-50*spring_k'], ['0'], ['0']]
    Sphere 1
      Radius: radius
      Position: ['0', '0', '-radius']
      Orientation: ['-phi/radius', '0', '0']
      Coupling matrix C_H:
[['1', '0', '0', '0'], ['0', '1', '0', 'sin(phi/radius)'], ['0', '0', '1', 'cos(phi/radius)'], ['0', '0', '0', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['50*spring_k'], ['0'], ['0']]


### Parameters of flow and gravity field

In [6]:
# Calculating force to have a swimming velocity of 1
tensors = myrigidbody.compute_fast_tensors(dofs=jnp.zeros(myrigidbody.Ndof))
FORCE_INTENSITY=1/tensors.M_H[2,-1]

In [ ]:
# Create a TGV flow with with max vorticy = 1
FLOW = sm.taylor_green_flow(omega=2)

# Create a gravity field
GRAVITY = sm.gravity_field(g=50.0)

# Create an active force scalar
FORCE = sm.constant_scalar(value=FORCE_INTENSITY)

In [8]:
# parameters of time integration
FINAL_TIME = 4 * jnp.pi
DT = 0.1
N_TRAJECTORIES = 15

Y_INIT_POSITIONS = np.linspace(np.pi/N_TRAJECTORIES/2, 2*np.pi, N_TRAJECTORIES, endpoint=False)
N_DT = int(FINAL_TIME / DT)

In [9]:
ROLLOUT = sm.FlowBodyRollout(
    soft_body = mybody,
    flow      = FLOW,
    input_map = {"gravity": GRAVITY, "active_force": FORCE},
)

### Objective and optimization

In [ ]:
def my_objective(rollout, design):
    tensors    = mybody.compute_tensors(design=design)
    force_norm = 1.0 / tensors.M_H[2, 3]
    FORCE.update_params(force_norm)

    # One rollout per initial position, average Z velocity
    velocities = jnp.stack([
        rollout.rollout(
            dt          = DT,
            n_steps     = N_DT,
            init_position = [0, x_init, 0],
            design      = design,
        )[0][-1, 2] / FINAL_TIME
        for x_init in Y_INIT_POSITIONS
    ])
    return jnp.mean(velocities)

In [11]:
myoptimizer = sm.FlowBodyOptimizer(ROLLOUT, my_objective, optimizer=optax.adam(0.001))

optimal_design = myoptimizer.run(
    init_design = 0.5 * jnp.ones(mybody.Ndesign),
    n_steps     = 1000,
    print_every = 100,
    clip_min    = 0.01,
    clip_max    = 1.0,
    grad_clip   = 0.1,
    maximize    = True,
)

step    0  objective=0.8912  |grad|=0.1000  design0=0.50  design1=0.50
step  100  objective=1.0833  |grad|=0.1000  design0=0.40  design1=0.40
step  200  objective=1.2125  |grad|=0.0512  design0=0.31  design1=0.30
step  300  objective=1.2126  |grad|=0.0020  design0=0.31  design1=0.29
step  400  objective=1.2126  |grad|=0.0005  design0=0.31  design1=0.29
step  500  objective=1.2126  |grad|=0.0001  design0=0.31  design1=0.29
step  600  objective=1.2126  |grad|=0.0000  design0=0.31  design1=0.29
step  700  objective=1.2126  |grad|=0.0000  design0=0.31  design1=0.29
step  800  objective=1.2126  |grad|=0.0000  design0=0.31  design1=0.29
step  900  objective=1.2126  |grad|=0.0000  design0=0.31  design1=0.29
step  999  objective=1.2126  |grad|=0.0000  design0=0.31  design1=0.29


### Surfer functions

In [12]:
tensors = mybody.compute_fast_tensors(dofs=jnp.zeros(mybody.Ndof), design=optimal_design)

TAU_ALIGN = 0.5/tensors.M_H[3,1]/50
print(TAU_ALIGN)

0.14060389


In [13]:
from jax.scipy.linalg import expm

E_Z = jnp.array([0.0, 0.0, 1.0])   # reference "upward" direction

def _rollout_surfer(tau, x_init):
    """Surfer agent: swims at speed 1, aligns with z @ exp(tau * grad_u)."""

    position = jnp.array([np.pi/2, x_init, 0.0])
    hatp     = E_Z.copy()   # initial swimming direction: upward

    def step(carry, t):
        position, hatp = carry
        time = t * DT

        u_lab              = FLOW.velocity(position, time)
        omega_lab, _       = FLOW.omega_rate_of_strain(position, time)
        grad_u             = FLOW.gradient(position, time)   # 3×3 matrix

        # --- Preferred direction ---
        n    = E_Z @ expm(tau * grad_u)         # deform current heading
        hatn = n / jnp.linalg.norm(n)           # normalize

        # --- hatp ODE (on the unit sphere) ---
        #   d hatp/dt = omega × hatp + (hatn - (hatn·hatp) hatp) / (2 talign)
        vorticity_term  = jnp.cross(omega_lab, hatp)
        alignment_term  = (hatn - jnp.dot(hatn, hatp) * hatp) / (2.0 * TAU_ALIGN)
        dhatp           = vorticity_term + alignment_term

        # --- Position ODE: advected by flow + swimming at speed 1 ---
        dpos = u_lab + hatp

        # --- RK2 ---
        pos_mid  = position + DT * dpos  / 2
        hatp_mid = hatp     + DT * dhatp / 2
        hatp_mid = hatp_mid / jnp.linalg.norm(hatp_mid)   # keep on sphere

        u_mid            = FLOW.velocity(pos_mid, time + DT/2)
        omega_mid, _     = FLOW.omega_rate_of_strain(pos_mid, time + DT/2)
        grad_u_mid       = FLOW.gradient(pos_mid, time + DT/2)

        n_mid    = E_Z @ expm(tau * grad_u_mid)
        hatn_mid = n_mid / jnp.linalg.norm(n_mid)

        dhatp_mid = (jnp.cross(omega_mid, hatp_mid)
                     + (hatn_mid - jnp.dot(hatn_mid, hatp_mid) * hatp_mid) / (2.0 * TAU_ALIGN))
        dpos_mid  = u_mid + hatp_mid

        position = position + DT * (dpos + dpos_mid) / 2
        hatp     = hatp     + DT * (dhatp + dhatp_mid) / 2
        hatp     = hatp     / jnp.linalg.norm(hatp)   # project back to S²

        return (position, hatp), position   # save position for trajectory

    (position, hatp), positions = jax.lax.scan(
        step, (position, hatp), jnp.arange(N_DT)
    )

    return position[2] / FINAL_TIME, positions   # mean Z velocity + full path


def mean_velocity_surfer(tau):
    velocities  = jnp.stack([_rollout_surfer(tau, x)[0] for x in Y_INIT_POSITIONS])
    return -jnp.mean(velocities)

grad_fn_surfer = jax.jit(jax.value_and_grad(mean_velocity_surfer))

In [14]:
def gradient_descent(
    grad_fn,
    initial_design,
    lr           = 0.05,
    n_steps      = 1000,
    print_every  = 100,
    clip_min     = 0.0,
    clip_max     = 1.0,
    grad_clip    = 1.0,    # max gradient norm — set None to disable
):
    design = initial_design
    design_keys = mybody.design_variables
    loss, grad = None, None
    best_design, best_velocity = initial_design, -jnp.inf

    for step in range(n_steps):
        loss, grad = grad_fn(design)

        # --- NaN guard: stop and revert to best if things blow up ---
        if jnp.isnan(loss) or jnp.any(jnp.isnan(grad)):
            print(
                f"  !! NaN detected at step {step}, stopping. "
                f"Best was step with velocity={float(best_velocity):.5f}"
            )
            return best_design, float(best_velocity)

        # --- Gradient clipping ---
        if grad_clip is not None:
            grad_norm = jnp.linalg.norm(grad)
            grad      = jnp.where(grad_norm > grad_clip, grad * grad_clip / grad_norm, grad)

        design = design - lr * grad
        design = jnp.clip(design, clip_min, clip_max)

        velocity = float(-loss)
        if velocity > best_velocity:
            best_velocity = velocity
            best_design   = design

        if step % print_every == 0 or step == n_steps - 1:
            if len(design_keys)==len(design):
                params_str = "  ".join(f"{k}={float(v):.4f}" for k, v in zip(design_keys, design, strict=False))
            else:
                params_str = "  ".join(f"d{i}={float(v):.4f}" for i, v in enumerate(design))
            print(f"step {step:4d}  velocity={velocity:.5f}  "
                f"|grad|={float(jnp.linalg.norm(grad)):.5f}  {params_str}")

    print("\nOptimal design:")
    for k, v in zip(design_keys, best_design, strict=False):
        print(f"  {k}: {float(v):.4f}  (init: {float(initial_design[list(design_keys).index(k)]):.4f})")
    print(f"Mean velocity: {float(best_velocity):.5f}")

    return best_design, float(best_velocity)

### Surfer optimization

In [15]:
initial_guess = 0.5 * jnp.ones(1)

optimal_surfer, optimal_velocity = gradient_descent(
    grad_fn_surfer,
    initial_guess,
    lr          = 0.01,
    n_steps     = 1000,
    print_every = 100,
    clip_min=0.01,
    clip_max=2.0,
)

step    0  velocity=1.17307  |grad|=0.24727  d0=0.5025
step  100  velocity=1.18893  |grad|=0.04150  d0=0.6122
step  200  velocity=1.18950  |grad|=0.01015  d0=0.6341
step  300  velocity=1.18953  |grad|=0.00263  d0=0.6396
step  400  velocity=1.18954  |grad|=0.00069  d0=0.6410
step  500  velocity=1.18954  |grad|=0.00018  d0=0.6414
step  600  velocity=1.18954  |grad|=0.00005  d0=0.6415
step  700  velocity=1.18954  |grad|=0.00001  d0=0.6415
step  800  velocity=1.18954  |grad|=0.00000  d0=0.6416
step  900  velocity=1.18954  |grad|=0.00000  d0=0.6416
step  999  velocity=1.18954  |grad|=0.00000  d0=0.6416

Optimal design:
  radius: 0.6415  (init: 0.5000)
Mean velocity: 1.18954


### Simulations for plots

In [16]:
DT = 0.1
N_DT = int(FINAL_TIME / DT)

tensors = mybody.compute_fast_tensors(dofs=jnp.zeros(mybody.Ndof), design=optimal_design)
FORCE_INTENSITY = 1.0/tensors.M_H[2,-1]
FORCE = sm.constant_scalar(value=FORCE_INTENSITY)

In [17]:
surfer_trajectories = []

for yinit in Y_INIT_POSITIONS:
    v, p = _rollout_surfer(tau=optimal_surfer, x_init=yinit)
    print("New initial position:", yinit, ", velocity:", v)
    surfer_trajectories.append(p[:, np.newaxis, :])

end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in surfer_trajectories]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/FINAL_TIME:.3g}")
surfer_poss = [jnp.array([t[0] for t in traj]) for traj in surfer_trajectories]

New initial position: 0.10471975511965977 , velocity: 1.0021665
New initial position: 0.5166174585903216 , velocity: 1.1882607
New initial position: 0.9285151620609834 , velocity: 1.2419479
New initial position: 1.340412865531645 , velocity: 1.2604989
New initial position: 1.7523105690023069 , velocity: 1.2614446
New initial position: 2.164208272472969 , velocity: 1.2457885
New initial position: 2.5761059759436304 , velocity: 1.1954705
New initial position: 2.9880036794142923 , velocity: 0.9988653
New initial position: 3.3999013828849542 , velocity: 1.0686227
New initial position: 3.811799086355616 , velocity: 1.214895
New initial position: 4.223696789826278 , velocity: 1.2516743
New initial position: 4.6355944932969395 , velocity: 1.262677
New initial position: 5.0474921967676005 , velocity: 1.257673
New initial position: 5.459389900238262 , velocity: 1.2332512
New initial position: 5.871287603708924 , velocity: 1.159822
End Z positions: ['4.01π', '4.75π', '4.97π', '5.04π', '5.05π', '

In [18]:
mybody.set_design_defaults(new_design=optimal_design)
myrollout = sm.FlowBodyRollout(
    soft_body = mybody,
    flow      = FLOW,
    input_map = {"gravity": GRAVITY, "active_force": FORCE},
)

poss = []

for x_init in Y_INIT_POSITIONS:
    print("New initial position", x_init)
    pos, ori, dof = myrollout.rollout(
            dt          = DT,
            n_steps     = N_DT,
            init_position = jnp.array([jnp.pi/2, x_init, 0.0]),
        )
    poss.append(pos)

end_z_positions = [float(p[-1, 2]) for p in poss]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/FINAL_TIME:.3g}")

OLD default param values: ['radius', 'spring_k'] [ 0.33  0.5 ]
NEW default param values: ['radius', 'spring_k'] [ 0.311  0.289]
New initial position 0.10471975511965977
New initial position 0.5166174585903216
New initial position 0.9285151620609834
New initial position 1.340412865531645
New initial position 1.7523105690023069
New initial position 2.164208272472969
New initial position 2.5761059759436304
New initial position 2.9880036794142923
New initial position 3.3999013828849542
New initial position 3.811799086355616
New initial position 4.223696789826278
New initial position 4.6355944932969395
New initial position 5.0474921967676005
New initial position 5.459389900238262
New initial position 5.871287603708924
End Z positions: ['2.81π', '3.88π', '4.71π', '5.09π', '5.48π', '5.61π', '5.52π', '4.78π', '5.03π', '5.58π', '5.6π', '5.41π', '4.98π', '4.56π', '3.73π']
Mean velocity: 1.21


In [19]:
myrigidbody.set_design_defaults(new_design=[optimal_design[0]])
myrigidrollout = sm.FlowBodyRollout(
    soft_body = myrigidbody,
    flow      = FLOW,
    input_map = {"gravity": GRAVITY, "active_force": FORCE},
)

rigid_poss = []

for x_init in Y_INIT_POSITIONS:
    print("New initial position", x_init)
    pos, ori, dof = myrigidrollout.rollout(
            dt          = DT,
            n_steps     = N_DT,
            init_position = jnp.array([jnp.pi/2, x_init, 0.0]),
        )
    rigid_poss.append(pos)

end_z_positions = [float(p[-1, 2]) for p in rigid_poss]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/FINAL_TIME:.3g}")

OLD default param values: ['radius'] [ 0.5]
NEW default param values: ['radius'] [ 0.311]
New initial position 0.10471975511965977
New initial position 0.5166174585903216
New initial position 0.9285151620609834
New initial position 1.340412865531645
New initial position 1.7523105690023069
New initial position 2.164208272472969
New initial position 2.5761059759436304
New initial position 2.9880036794142923
New initial position 3.3999013828849542
New initial position 3.811799086355616
New initial position 4.223696789826278
New initial position 4.6355944932969395
New initial position 5.0474921967676005
New initial position 5.459389900238262
New initial position 5.871287603708924
End Z positions: ['0.25π', '0.0951π', '0.215π', '2.17π', '2.09π', '4.51π', '5.41π', '3.45π', '3.33π', '5.77π', '4.36π', '2.08π', '2.22π', '0.238π', '0.101π']
Mean velocity: 0.605


### Preparing the figure of trajectory

In [ ]:
# --- Helpers to build π-spaced ticks ---
def pi_ticks(lo, hi, step_pi=1):
    """Return (tickvals, ticktext) with spacing of step_pi * π."""
    import math
    n_lo = math.floor(lo / (step_pi * np.pi))
    n_hi = math.ceil(hi / (step_pi * np.pi))
    vals, texts = [], []
    for n in range(n_lo, n_hi + 1):
        v = n * step_pi * np.pi
        vals.append(v)
        k = n * step_pi          # multiple of π
        if k == 0:
            texts.append("0")
        elif k == 1:
            texts.append(r"$\pi$")
        elif k == -1:
            texts.append(r"$-\pi$")
        else:
            texts.append(rf"${k}\pi$")
    return vals, texts


def bounds_2pi(lo, hi, step_pi=1):
    """Round lo down and hi up to the nearest multiple of step_pi*π."""
    unit = step_pi * np.pi
    return np.floor(lo / unit) * unit, np.ceil(hi / unit) * unit

In [ ]:
colors = [BLUEACCENT, REDACCENT, GREY]
panel_titles = ("rigid", "soft", "surfer")
trajectory_sets = (rigid_poss, poss, surfer_poss)

# --- Compute global bounds across ALL trajectories ---
all_y = np.concatenate([p[:, 1] for p in poss + rigid_poss + surfer_poss])
all_z = np.concatenate([p[:, 2] for p in poss + rigid_poss + surfer_poss])

Y_MIN, Y_MAX = bounds_2pi(all_y.min(), all_y.max())
Z_MIN, Z_MAX = bounds_2pi(all_z.min(), all_z.max())

GRID_STEP = 1
y_tickvals, y_ticktext = pi_ticks(Y_MIN, Y_MAX, step_pi=GRID_STEP)
z_tickvals, z_ticktext = pi_ticks(Z_MIN, Z_MAX, step_pi=GRID_STEP)

# Aspect: y-extent vs z-extent — the figure is wide if z is short.
aspect_total = 3 * (Y_MAX - Y_MIN) / (Z_MAX - Z_MIN)

fig, axes = figstyle.subplots(
    size="full", aspect=aspect_total, ncols=3, sharey=True,
)

# Streamlines: psi = 0.5 * sin(y) * sin(z) for the Taylor-Green flow.
ny, nz = 300, 300
y_grid = np.linspace(Y_MIN, Y_MAX, ny)
z_grid = np.linspace(Z_MIN, Z_MAX, nz)
Y, Z = np.meshgrid(y_grid, z_grid)
PSI = 0.5 * np.sin(Y) * np.sin(Z)

for k, (ax, title, traj_set) in enumerate(
    zip(axes, panel_titles, trajectory_sets, strict=True)
):
    ax.contour(Y, Z, PSI, levels=8, colors=GREY, linewidths=0.8)
    for positions in traj_set:
        y_pos = np.array(positions[:, 1])
        z_pos = np.array(positions[:, 2])
        ax.plot(y_pos, z_pos, color=colors[k], linewidth=1.5)
        ax.plot(y_pos[0], z_pos[0], marker="o", color=colors[k], markersize=4)
        ax.plot(y_pos[-1], z_pos[-1], marker="s", color=colors[k], markersize=4)

    ax.set_title(title)
    ax.set_xlim(Y_MIN, Y_MAX)
    ax.set_ylim(Z_MIN, Z_MAX)
    ax.set_aspect("equal")
    ax.set_xticks(y_tickvals, y_ticktext)
    if k == 0:
        ax.set_yticks(z_tickvals, z_ticktext)

fig

### Figure

In [ ]:
fig